# 🎙️ Tetrax ASR: Fine-Tuning Whisper Medium on IndicVoices for Marathi + Hindi

Welcome to the official **Tetrax ASR Bilingual Fine-Tuning Notebook**.

This notebook trains and adapts **`openai/whisper-medium`** (769M multilingual parameters) using **PEFT / LoRA (Low-Rank Adaptation)** on the **AI4Bharat IndicVoices** dataset to build a high-accuracy speech recognition model for **Marathi (`mr`)** and **Hindi (`hi`)**.

### 🚀 Memory Optimization for Free Google Colab T4 GPU (15GB VRAM):
- Fine-tunes **`openai/whisper-medium`** using parameter-efficient LoRA (`r=32`, `lora_alpha=64`).
- Trains only ~1.1% of parameters (~8.3M params) while freezing the core 769M acoustic backbone.
- `per_device_train_batch_size = 1` with `gradient_accumulation_steps = 8` (effective batch size = 8).
- `fp16 = True` and `gradient_checkpointing = True` to fit comfortably in 15GB VRAM without Out-Of-Memory (OOM) errors.
- Evaluates both **WER** (Word Error Rate) and **CER** (Character Error Rate for Devanagari script).

---

## Step 1: Check GPU & Install Dependencies

In [ ]:
# Verify GPU allocation (T4 GPU with ~15GB VRAM recommended)
!nvidia-smi

# Install required fine-tuning and evaluation libraries
!pip install -q transformers datasets peft accelerate evaluate jiwer soundfile librosa bitsandbytes tensorboard

## Step 2: Mount Google Drive
Mount your drive to access your manually uploaded IndicVoices dataset in `/content/drive/MyDrive/Tetrax-ASR/data/` and permanently save your fine-tuned Whisper Medium checkpoints.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_DATA_DIR = "/content/drive/MyDrive/Tetrax-ASR/data"
CHECKPOINT_DIR = "/content/drive/MyDrive/Tetrax-ASR/checkpoints/tetrax-whisper-medium-marathi-hindi"
os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"✅ Google Drive mounted successfully.")
print(f"📁 Data directory: {DRIVE_DATA_DIR}")
print(f"💾 Checkpoints directory: {CHECKPOINT_DIR}")

## Step 3: Load AI4Bharat IndicVoices Dataset (Marathi + Hindi)
This cell automatically scans your Google Drive (`/content/drive/MyDrive/Tetrax-ASR/data/`) or `/content/` for manually placed parquet files:
1. Checks subfolders: `data/marathi/` and `data/hindi/`
2. Checks filename patterns: `*marathi*.parquet` and `*hindi*.parquet` (or generic `*.parquet`)
3. If local files are not found, it downloads directly from Hugging Face Hub (`ai4bharat/IndicVoices`).
4. Creates a balanced, stratified 80% Train / 10% Validation / 10% Test split across Marathi and Hindi.

In [ ]:
import glob
import os
from datasets import load_dataset, Audio, DatasetDict, concatenate_datasets

DRIVE_DATA = "/content/drive/MyDrive/Tetrax-ASR/data"
LOCAL_DATA = "/content"

def find_lang_parquets(lang_name, lang_code):
    patterns = [
        f"{DRIVE_DATA}/{lang_name}/*.parquet",
        f"{DRIVE_DATA}/{lang_name}/**/*.parquet",
        f"{DRIVE_DATA}/{lang_code}/*.parquet",
        f"{DRIVE_DATA}/*{lang_name}*.parquet",
        f"{DRIVE_DATA}/*{lang_code}*.parquet",
        f"{LOCAL_DATA}/{lang_name}/*.parquet",
        f"{LOCAL_DATA}/*{lang_name}*.parquet",
    ]
    files = []
    for p in patterns:
        files.extend(glob.glob(p, recursive=True))
    return sorted(list(dict.fromkeys(files)))

mr_files = find_lang_parquets("marathi", "mr")
hi_files = find_lang_parquets("hindi", "hi")

print(f"Found {len(mr_files)} Marathi parquet file(s) in Drive: {mr_files}")
print(f"Found {len(hi_files)} Hindi parquet file(s) in Drive: {hi_files}")

# Load Marathi
if mr_files:
    print(f"Loading Marathi from local files...")
    ds_mr = load_dataset("parquet", data_files=mr_files, split="train")
else:
    print("Loading Marathi from Hugging Face Hub (ai4bharat/IndicVoices, 'marathi')...")
    ds_mr = load_dataset("ai4bharat/IndicVoices", "marathi", split="train")

if "language" not in ds_mr.column_names:
    ds_mr = ds_mr.add_column("language", ["mr"] * len(ds_mr))

# Load Hindi
if hi_files:
    print(f"Loading Hindi from local files...")
    ds_hi = load_dataset("parquet", data_files=hi_files, split="train")
else:
    print("Loading Hindi from Hugging Face Hub (ai4bharat/IndicVoices, 'hindi')...")
    ds_hi = load_dataset("ai4bharat/IndicVoices", "hindi", split="train")

if "language" not in ds_hi.column_names:
    ds_hi = ds_hi.add_column("language", ["hi"] * len(ds_hi))

# Normalize audio columns with decode=False for speed & stability
for ds in [ds_mr, ds_hi]:
    for col in ["audio_filepath", "audio"]:
        if col in ds.column_names:
            ds = ds.cast_column(col, Audio(decode=False))
            break

# Combine into bilingual dataset
raw_dataset = concatenate_datasets([ds_mr, ds_hi])
print(f"\nTotal merged samples: {len(raw_dataset)} ({len(ds_mr)} Marathi, {len(ds_hi)} Hindi)")

# Stratified 80/10/10 split across languages
raw_dataset = raw_dataset.class_encode_column("language")
first_split = raw_dataset.train_test_split(test_size=0.20, seed=42, stratify_by_column="language")
train_ds = first_split["train"]

second_split = first_split["test"].train_test_split(test_size=0.50, seed=42, stratify_by_column="language")
val_ds = second_split["train"]
test_ds = second_split["test"]

dataset_dict = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})

print(f"\n✅ Balanced splits created:")
print(f"  • Train samples: {len(train_ds)}")
print(f"  • Validation samples: {len(val_ds)}")
print(f"  • Test samples: {len(test_ds)}")

### Inspect Sample Marathi and Hindi Audio & Devanagari Transcripts

In [ ]:
import io
import soundfile as sf
import IPython.display as ipd

for idx in [0, len(train_ds) // 2]:
    sample = train_ds[idx]
    lang_val = "mr" if sample.get("language") == 0 else "hi"
    text_val = sample.get("text", sample.get("normalized_text", ""))
    print(f"\n--- Sample #{idx} [{lang_val.upper()}] ---")
    print("Transcript:", text_val)
    print("Duration:", sample.get("duration"), "sec")

    audio_entry = sample.get("audio_filepath") or sample.get("audio")
    if isinstance(audio_entry, dict) and "bytes" in audio_entry and audio_entry["bytes"]:
        audio_data, sr = sf.read(io.BytesIO(audio_entry["bytes"]))
        ipd.display(ipd.Audio(audio_data, rate=sr))

## Step 4: Setup Multilingual Whisper Medium Processor
We initialize the WhisperProcessor for **`openai/whisper-medium`** configured for multilingual transcription.

In [ ]:
from transformers import WhisperProcessor

MODEL_ID = "openai/whisper-medium"
print(f"Initializing WhisperProcessor for {MODEL_ID}...")

processor = WhisperProcessor.from_pretrained(MODEL_ID, language="mr", task="transcribe")
print("✅ Whisper Medium processor initialized.")

## Step 5: Bilingual Preprocessing & Dynamic Language Tokenization
Computes 80-channel log-Mel spectrograms from 16kHz audio, and dynamically conditions the tokenizer tokens for each sample (`<|mr|>` for Marathi, `<|hi|>` for Hindi).

In [ ]:
import librosa
import numpy as np

def prepare_bilingual_dataset(batch):
    # 1. Decode audio bytes
    audio_entry = batch.get("audio_filepath") or batch.get("audio")
    if isinstance(audio_entry, dict) and "bytes" in audio_entry and audio_entry["bytes"]:
        audio_arr, orig_sr = sf.read(io.BytesIO(audio_entry["bytes"]), dtype="float32")
    elif isinstance(audio_entry, dict) and "array" in audio_entry and audio_entry["array"] is not None:
        audio_arr = np.array(audio_entry["array"], dtype=np.float32)
        orig_sr = audio_entry.get("sampling_rate", 16000)
    else:
        raise ValueError("Unrecognized audio column format")

    if audio_arr.ndim > 1:
        audio_arr = audio_arr.mean(axis=1)
    if orig_sr != 16000:
        audio_arr = librosa.resample(audio_arr, orig_sr=orig_sr, target_sr=16000)

    # 2. Extract log-Mel spectrogram
    batch["input_features"] = processor.feature_extractor(
        audio_arr,
        sampling_rate=16000
    ).input_features[0]

    # 3. Dynamic language prompt prefixing (mr vs hi)
    sample_lang = "mr" if batch.get("language") in (0, "mr", "marathi") else "hi"
    processor.tokenizer.set_prefix_tokens(language=sample_lang, task="transcribe")
    text_content = batch.get("text", batch.get("normalized_text", ""))
    batch["labels"] = processor.tokenizer(text_content).input_ids
    return batch

print("Mapping features and tokenizing labels across splits...")
tokenized_train = dataset_dict["train"].map(prepare_bilingual_dataset, remove_columns=dataset_dict["train"].column_names, num_proc=2)
tokenized_val = dataset_dict["validation"].map(prepare_bilingual_dataset, remove_columns=dataset_dict["validation"].column_names, num_proc=2)
print("✅ Bilingual preprocessing finished successfully!")

## Step 6: Data Collator and Devanagari Evaluation Metrics (WER + CER)
We compute both **Word Error Rate (WER)** and **Character Error Rate (CER)** to accurately measure recognition performance on Devanagari text.

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    cer = 100 * cer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer, "cer": cer}

## Step 7: Load Whisper Medium (769M) & Apply PEFT LoRA with Gradient Checkpointing

In [ ]:
from transformers import WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model

print(f"Loading WhisperForConditionalGeneration ({MODEL_ID}) in FP16...")
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

model.config.use_cache = False
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# Enable gradient checkpointing to fit Whisper-Medium within Colab's 15GB T4 GPU
model.gradient_checkpointing_enable()

# LoRA Configuration targeting attention projections
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
    lora_dropout=0.05,
    bias="none"
)

model = get_peft_model(model, peft_config)
print("\n📊 Trainable Parameters Summary:")
model.print_trainable_parameters()

## Step 8: Fine-Tune with Seq2SeqTrainer (T4 GPU Memory Optimized)
- `per_device_train_batch_size = 1`
- `gradient_accumulation_steps = 8` (effective batch size = 8)
- `gradient_checkpointing = True`
- `fp16 = True`

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
import inspect

OUTPUT_DIR = "/content/tetrax-whisper-medium-marathi-hindi"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    warmup_steps=100,
    num_train_epochs=3,
    eval_strategy="steps",
    fp16=True,
    per_device_eval_batch_size=1,
    generation_max_length=225,
    save_steps=100,
    eval_steps=100,
    logging_steps=25,
    gradient_checkpointing=True,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    save_total_limit=2,
    remove_unused_columns=False,
    label_names=["labels"],
)

trainer_kwargs = {
    "args": training_args,
    "model": model,
    "train_dataset": tokenized_train,
    "eval_dataset": tokenized_val,
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
}

if "processing_class" in inspect.signature(Seq2SeqTrainer.__init__).parameters:
    trainer_kwargs["processing_class"] = processor.feature_extractor
else:
    trainer_kwargs["tokenizer"] = processor.feature_extractor

trainer = Seq2SeqTrainer(**trainer_kwargs)

print("🚀 Starting Whisper Medium bilingual fine-tuning on Marathi + Hindi...")
trainer.train()

## Step 9: Save Fine-Tuned LoRA Adapter & Export to Google Drive & ZIP

In [ ]:
import shutil
import os
from google.colab import files

OUTPUT_DIR = "/content/tetrax-whisper-medium-marathi-hindi"
ZIP_NAME = "tetrax-whisper-medium-marathi-hindi.zip"

# 1. Save adapter & processor
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"✅ Saved fine-tuned LoRA adapter to: {OUTPUT_DIR}")

# 2. Package into a compressed ZIP file
shutil.make_archive("tetrax-whisper-medium-marathi-hindi", "zip", OUTPUT_DIR)
zip_size_mb = os.path.getsize(ZIP_NAME) / (1024 * 1024) if os.path.exists(ZIP_NAME) else 0.0
print(f"📦 Successfully created ZIP archive: {ZIP_NAME} ({zip_size_mb:.2f} MB)")

# 3. Save both ZIP and uncompressed folder to Google Drive
DRIVE_DIR = "/content/drive/MyDrive/Tetrax-ASR/checkpoints"
try:
    os.makedirs(DRIVE_DIR, exist_ok=True)
    drive_zip_path = os.path.join(DRIVE_DIR, ZIP_NAME)
    shutil.copy2(ZIP_NAME, drive_zip_path)
    print(f"💾 SAVED ZIP TO GOOGLE DRIVE: {drive_zip_path}")

    drive_folder_path = os.path.join(DRIVE_DIR, "tetrax-whisper-medium-marathi-hindi")
    shutil.copytree(OUTPUT_DIR, drive_folder_path, dirs_exist_ok=True)
    print(f"📁 SAVED FOLDER TO GOOGLE DRIVE: {drive_folder_path}")
except Exception as e:
    print(f"⚠️ Notice copying to Drive ({e}). Make sure drive.mount('/content/drive') was run in Step 2.")

# 4. Trigger direct browser download
print("⬇️ Triggering direct browser download of ZIP...")
try:
    files.download(ZIP_NAME)
except Exception as e:
    print("Browser download notice:", e)

## 🎉 Deployment into your Tetrax ASR Project

Once downloaded:
1. Extract `tetrax-whisper-medium-marathi-hindi.zip`.
2. Place the folder into `Tetrax-ASR/checkpoints/tetrax-whisper-medium-marathi-hindi/`.
3. Start your Tetrax ASR service:
   ```bash
   python -m uvicorn tetrax_asr.api.main:app --host 0.0.0.0 --port 8000
   ```
4. The Tetrax ASR engine automatically detects and mounts your fine-tuned bilingual Marathi + Hindi LoRA adapter for both REST API (`POST /v1/audio/transcriptions` with `language='mr'` or `'hi'`) and live streaming captions!